<a href="https://colab.research.google.com/github/jaunlois/msxwarrtools/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import subprocess
import traceback
import shutil
import importlib

# Force reload torch if it's already in a partially initialized state
if 'torch' in sys.modules:
    try:
        importlib.reload(sys.modules['torch'])
    except Exception:
        pass

# Ensure we are in /content
os.chdir('/content')

# Remove any existing repo and clone fresh
repo_path = "/content/msxwarrtools"
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "-b",
        "develop",
        "https://github.com/jaunlois/msxwarrtools.git",
        repo_path,
    ],
    check=True,
)

# Note: Skipping pip install to avoid circular import errors with torch in an active session.

# Change directory
os.chdir(repo_path)

# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Model setup
MODEL_ID = "Nasim435/Qwen-3B-Automotive-4000"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

# Run a test generation
try:
    msgs = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "hi"},
    ]
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text, return_tensors="pt", return_attention_mask=True
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.3,
            pad_token_id=tokenizer.pad_token_id,
        )

    reply = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

    print("===WORKING===")
    print(repr(reply[:200]))

except Exception:
    print("===BROKEN===")
    print(traceback.format_exc())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

===BROKEN===
Traceback (most recent call last):
  File "/tmp/ipykernel_4981/3745931829.py", line 74, in <cell line: 0>
    text = tokenizer.apply_chat_template(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 3112, in apply_chat_template
    chat_template = self.get_chat_template(chat_template, tools)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 3294, in get_chat_template
    raise ValueError(
ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating



In [5]:
from transformers import AutoTokenizer
import torch, traceback

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model.config.pad_token_id = tokenizer.pad_token_id

print("Chat template ready:", bool(tokenizer.chat_template))

try:
      msgs = [{"role":"system","content":"You are helpful."},{"role":"user","content":"hi"}]
      text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
      inputs = tokenizer(text, return_tensors="pt", return_attention_mask=True).to(model.device)
      with torch.no_grad():
          out = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=50,
  do_sample=True, temperature=0.3, pad_token_id=tokenizer.pad_token_id)
      reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
      print("===WORKING===")
      print(repr(reply[:200]))
except Exception:
      print("===BROKEN===")
      print(traceback.format_exc())

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Chat template ready: True
===WORKING===
'Hello! How can I assist you today?'


In [7]:
import os
os.environ["NGROK_AUTHTOKEN"] = "3E6J8SuOSTb1Yx2ISBl2OIG3emj_4EajUfsqBkwroxMEjHNRM"
os.chdir("/content/msxwarrtools")
!git pull
%run -i colab/serve.py

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.12 KiB | 1.12 MiB/s, done.
From https://github.com/jaunlois/msxwarrtools
   2bf01ad..e9dcc83  develop    -> origin/develop
Updating 2bf01ad..e9dcc83
Fast-forward
 colab/serve.py | 15 +++++++++++++++
 1 file changed, 15 insertions(+)

 Public URL:   https://unjustly-headfirst-conceal.ngrok-free.dev

 Paste into /consultant Settings on your Render app:
   API Base URL: https://unjustly-headfirst-conceal.ngrok-free.dev/v1
   Model:        qwen-automotive
   API Key:      anything-non-empty

